# Chapter 9: Bayes Filter (Recursive Estimation)

<a href="../lite/lab/index.html?path=ch09_bayes_filter.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

A robot is lost in a hallway with 3 doors. It has no idea where it is, so its belief is a
flat line. Then it sees a door. Suddenly, its belief has 3 peaks. It moves forward 2 meters
and sees another door. Now only 1 peak survives. In just two steps, the robot went from
completely lost to confidently located. That is the **Bayes filter**.

The Bayes filter is the foundation of nearly every estimation algorithm in robotics.
The Kalman filter, the particle filter, and grid based localization are all special cases.
This chapter builds the filter from scratch, one step at a time, so you can see exactly
how raw sensor data transforms into confident estimates.

```{admonition} What you will build
:class: tip

- Implement a complete Bayes filter for a robot localizing in a 1D hallway with doors
- Watch the robot go from completely lost (uniform belief) to confidently located in just a few steps
- Diagnose what happens when the motion model is wrong
- Implement forward-backward smoothing to improve past estimates using future data

**Real world application:** The Bayes filter is the ancestor of every localization algorithm. After this chapter, you will understand the predict-update cycle that drives Kalman filters, particle filters, and SLAM.
```

## 9.1 Belief Representation

A **belief** is a probability distribution over the robot's state. It encodes everything the
robot knows (and does not know) about where it is, given all past actions and observations:

$$\text{bel}(x_t) = p(x_t \mid z_{1:t},\, u_{1:t})$$

where $x_t$ is the state at time $t$, $z_{1:t}$ are all measurements, and $u_{1:t}$ are all
control inputs.

There are many ways to represent a belief. Each has strengths and trade offs:

| Representation | What it stores | Pros | Cons |
|---|---|---|---|
| **Point estimate** | Single value $\hat{x}$ | Simple, fast | No uncertainty info |
| **Gaussian** | Mean $\mu$, variance $\sigma^2$ | Compact, elegant math | Only unimodal |
| **Histogram** | Grid of probabilities | Handles multimodality | Scales poorly to high dims |
| **Particles** | Weighted samples | Flexible, nonparametric | Needs many samples |

Let us visualize all four for the same underlying scenario: a robot that might be near
position 3 or position 7 in a hallway.

In [ ]:
# ── PARAMETERS ── change these and re-run ────
peak1 = 3.0          # first possible position
peak2 = 7.0          # second possible position
sigma = 0.5          # spread of each mode
n_particles = 200    # number of particles
n_hist_bins = 50     # histogram cells

x = np.linspace(0, 10, 500)

# build a bimodal ground truth
true_pdf = 0.5 * stats.norm.pdf(x, peak1, sigma) + 0.5 * stats.norm.pdf(x, peak2, sigma)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

# 1. Point estimate
ax = axes[0]
ax.axvline(np.mean([peak1, peak2]), color='tomato', linewidth=2, label='$\\hat{x}$')
ax.fill_between(x, true_pdf, alpha=0.15, color='steelblue', label='true belief')
ax.set_title('Point Estimate', fontweight='bold')
ax.set_xlabel('position')
ax.legend(fontsize=8)

# 2. Gaussian
ax = axes[1]
mu_g = np.mean([peak1, peak2])
sigma_g = 2.5
gaussian_approx = stats.norm.pdf(x, mu_g, sigma_g)
ax.plot(x, gaussian_approx, color='tomato', linewidth=2, label='Gaussian fit')
ax.fill_between(x, true_pdf, alpha=0.15, color='steelblue', label='true belief')
ax.set_title('Gaussian', fontweight='bold')
ax.set_xlabel('position')
ax.legend(fontsize=8)

# 3. Histogram
ax = axes[2]
hist_edges = np.linspace(0, 10, n_hist_bins + 1)
hist_centers = 0.5 * (hist_edges[:-1] + hist_edges[1:])
hist_vals = 0.5 * stats.norm.pdf(hist_centers, peak1, sigma) + 0.5 * stats.norm.pdf(hist_centers, peak2, sigma)
hist_vals /= hist_vals.sum()  # normalize
ax.bar(hist_centers, hist_vals, width=hist_edges[1] - hist_edges[0], color='steelblue',
       edgecolor='white', linewidth=0.5, alpha=0.8)
ax.set_title('Histogram (Grid)', fontweight='bold')
ax.set_xlabel('position')

# 4. Particles
ax = axes[3]
rng = np.random.default_rng(42)
# sample from bimodal
choices = rng.choice([0, 1], size=n_particles)
particles = np.where(choices == 0,
                     rng.normal(peak1, sigma, n_particles),
                     rng.normal(peak2, sigma, n_particles))
weights = np.ones(n_particles) / n_particles
ax.scatter(particles, weights, s=8, color='tomato', alpha=0.6)
ax.fill_between(x, true_pdf / true_pdf.max() * weights.max(), alpha=0.15, color='steelblue')
ax.set_title('Particles', fontweight='bold')
ax.set_xlabel('position')

fig.suptitle('Four Ways to Represent a Belief', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print("Notice: the Gaussian approximation CANNOT capture two peaks.")
print("Histograms and particles handle multimodal beliefs naturally.")

**Key insight:** For the rest of this chapter, we use the **histogram** representation because
it makes the predict and update steps easiest to visualize. In later chapters, we will switch
to Gaussians (Kalman filter) and particles (particle filter).

## 9.2 Prediction Step

The **prediction step** incorporates the robot's motion. If the robot moves, its belief
about where it is should shift accordingly. But motion is never perfect, so the belief
also **spreads out** (uncertainty grows).

Mathematically, the predicted belief is:

$$\overline{\text{bel}}(x_t) = \int p(x_t \mid u_t,\, x_{t-1})\;\text{bel}(x_{t-1})\;dx_{t-1}$$

This integral says: "For each possible previous position $x_{t-1}$, figure out where the
robot might end up after action $u_t$, then sum up all those possibilities."

In our **discrete histogram** world, this becomes a convolution: shift the histogram by the
commanded motion, then blur it by the motion noise.

### Implementation

1. **Shift** the probability mass by the commanded motion (deterministic part)
2. **Blur** with a Gaussian kernel (stochastic part, representing noise)

In [ ]:
# ── PARAMETERS ── change these and re-run ────
n_cells = 20           # number of cells in the hallway
start_cell = 4         # robot starts here (0-indexed)
move_amount = 2        # commanded motion: move right 2 cells
move_noise = 1.0       # std dev of motion noise (in cells)


def predict(belief, shift, noise_std, n):
    """Prediction step: shift + blur the belief."""
    # shift (roll) the belief
    predicted = np.roll(belief, shift)
    # blur with Gaussian to model motion noise
    if noise_std > 0:
        predicted = gaussian_filter1d(predicted, sigma=noise_std, mode='wrap')
    # re-normalize
    predicted /= predicted.sum()
    return predicted


# initial belief: certain at start_cell
belief_before = np.zeros(n_cells)
belief_before[start_cell] = 1.0

# predict
belief_after = predict(belief_before, move_amount, move_noise, n_cells)

cells = np.arange(n_cells)
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

axes[0].bar(cells, belief_before, color='steelblue', edgecolor='white')
axes[0].set_title('Before Prediction (known position)', fontweight='bold')
axes[0].set_xlabel('cell')
axes[0].set_ylabel('probability')
axes[0].set_xticks(cells)

axes[1].bar(cells, belief_after, color='tomato', edgecolor='white')
axes[1].axvline(start_cell + move_amount, color='forestgreen', linestyle='--',
               linewidth=2, label=f'expected position ({start_cell + move_amount})')
axes[1].set_title(f'After Prediction (move right {move_amount}, noise={move_noise})', fontweight='bold')
axes[1].set_xlabel('cell')
axes[1].legend()
axes[1].set_xticks(cells)

plt.suptitle('Prediction Step: Shift + Blur', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Before: all probability at cell {start_cell}")
print(f"After:  peak near cell {start_cell + move_amount}, but spread out due to noise")
print(f"Total probability sums to {belief_after.sum():.4f} (normalized)")

**Try it:** increase `move_noise` and watch the belief spread out more. Set it to 0 and
the belief stays as a single spike, perfectly shifted. Real robots always have some noise,
so the prediction step always increases uncertainty.

## 9.3 Update Step (Measurement Incorporation)

The **update step** (also called the **correction step**) incorporates a sensor measurement.
It uses Bayes' rule to combine the predicted belief with the measurement likelihood:

$$\text{bel}(x_t) = \eta\;p(z_t \mid x_t)\;\overline{\text{bel}}(x_t)$$

where:
- $p(z_t \mid x_t)$ is the **likelihood**: how probable is this measurement if the robot is at $x_t$?
- $\eta$ is a normalizing constant so the result sums to 1
- $\overline{\text{bel}}(x_t)$ is the predicted belief from Section 9.2

The update is a **pointwise multiplication** followed by **normalization**. It is remarkably
simple, yet powerful.

In [ ]:
# ── PARAMETERS ── change these and re-run ────
n_cells = 20
door_positions = [5, 12, 18]   # cells that have doors
sensor_reliability = 0.85      # P(sense door | at door)
sensor_false_pos = 0.1         # P(sense door | no door)

# start with a uniform belief (robot is lost)
prior_belief = np.ones(n_cells) / n_cells


def compute_likelihood(n, door_pos, reliability, false_pos):
    """Likelihood of sensing a door at each cell."""
    lik = np.full(n, false_pos)
    for d in door_pos:
        if 0 <= d < n:
            lik[d] = reliability
    return lik


def update(belief, likelihood):
    """Update step: pointwise multiply and normalize."""
    updated = belief * likelihood
    updated /= updated.sum()  # normalize
    return updated


# robot senses: "I see a door"
likelihood = compute_likelihood(n_cells, door_positions, sensor_reliability, sensor_false_pos)
posterior = update(prior_belief, likelihood)

cells = np.arange(n_cells)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# prior
axes[0].bar(cells, prior_belief, color='steelblue', edgecolor='white')
axes[0].set_title('Prior Belief (uniform, robot is lost)', fontweight='bold')
axes[0].set_xlabel('cell')
axes[0].set_ylabel('probability')

# likelihood
axes[1].bar(cells, likelihood, color='orange', edgecolor='white')
for d in door_positions:
    axes[1].annotate('door', (d, likelihood[d]), ha='center', va='bottom', fontsize=8, fontweight='bold')
axes[1].set_title('Likelihood p(see door | x)', fontweight='bold')
axes[1].set_xlabel('cell')

# posterior
axes[2].bar(cells, posterior, color='forestgreen', edgecolor='white')
axes[2].set_title('Posterior Belief (after seeing door)', fontweight='bold')
axes[2].set_xlabel('cell')

plt.suptitle('Update Step: Prior \u00d7 Likelihood \u2192 Posterior', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print(f"The robot was completely lost (uniform belief).")
print(f"After sensing a door, probability concentrates at door cells: {door_positions}")
print(f"Three peaks appear because the robot does not yet know WHICH door it is at.")

### Predict + Update Together

The real power emerges when we chain prediction and update steps. After sensing a door
(three peaks), the robot moves and senses again. Let us watch ambiguity resolve.

In [ ]:
# ── PARAMETERS ── change these and re-run ────
n_cells = 20
door_positions = [5, 12, 18]
sensor_reliability = 0.85
sensor_false_pos = 0.1
motion_cmd = 3        # move right 3 cells
motion_noise = 0.8    # motion noise std

# step 0: uniform
bel = np.ones(n_cells) / n_cells
lik = compute_likelihood(n_cells, door_positions, sensor_reliability, sensor_false_pos)

fig, axes = plt.subplots(2, 3, figsize=(16, 7))
cells = np.arange(n_cells)

# -- TIME STEP 1 --
# update: sense door
bel = update(bel, lik)
axes[0, 0].bar(cells, bel, color='forestgreen', edgecolor='white')
axes[0, 0].set_title('t=1: Update (sense door)', fontweight='bold')
axes[0, 0].set_ylabel('probability')

# predict: move right
bel = predict(bel, motion_cmd, motion_noise, n_cells)
axes[0, 1].bar(cells, bel, color='tomato', edgecolor='white')
axes[0, 1].set_title(f't=1: Predict (move +{motion_cmd})', fontweight='bold')

# -- TIME STEP 2 --
# The robot is now near cells 8, 15, or 1 (wrapped). Suppose it senses a door again.
# Only cell 15+3=18 area has a door nearby at cell 18.
bel = update(bel, lik)
axes[0, 2].bar(cells, bel, color='forestgreen', edgecolor='white')
axes[0, 2].set_title('t=2: Update (sense door again)', fontweight='bold')

# predict again
bel = predict(bel, motion_cmd, motion_noise, n_cells)
axes[1, 0].bar(cells, bel, color='tomato', edgecolor='white')
axes[1, 0].set_title(f't=2: Predict (move +{motion_cmd})', fontweight='bold')
axes[1, 0].set_ylabel('probability')

# -- TIME STEP 3 --
bel = update(bel, lik)
axes[1, 1].bar(cells, bel, color='forestgreen', edgecolor='white')
axes[1, 1].set_title('t=3: Update (sense door)', fontweight='bold')

bel = predict(bel, motion_cmd, motion_noise, n_cells)
axes[1, 2].bar(cells, bel, color='tomato', edgecolor='white')
axes[1, 2].set_title(f't=3: Predict (move +{motion_cmd})', fontweight='bold')

for ax in axes.flat:
    ax.set_xlabel('cell')
    ax.set_ylim(0, None)

plt.suptitle('Predict/Update Cycle: Ambiguity Resolves Over Time', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print("Watch how the three peaks gradually collapse into one dominant peak.")
print("Each predict step spreads uncertainty; each update step sharpens it.")

## 9.4 Filtering vs Smoothing

So far we have run the Bayes filter **forward in time**. At each step $t$, the filter
uses only data from times $1, 2, \ldots, t$. This is called **filtering**.

**Smoothing** goes further: it estimates the state at time $t$ using **all** data, including
future observations from times $t+1, t+2, \ldots, T$. Smoothing always produces beliefs
that are at least as sharp as filtering, because it has strictly more information.

The **forward backward algorithm** works as follows:
1. Run the Bayes filter forward to get $\text{bel}_{\text{fwd}}(x_t)$ for each $t$
2. Run a backward pass from $t = T$ to $t = 1$
3. Combine forward and backward messages at each $t$

$$\text{bel}_{\text{smooth}}(x_t) \propto \text{bel}_{\text{fwd}}(x_t) \cdot \beta_t(x_t)$$

where $\beta_t$ is the backward message carrying information from future observations.

In [ ]:
# ── PARAMETERS ── change these and re-run ────
n_cells = 20
n_steps = 12                   # total number of time steps
door_positions = [3, 10, 16]
motion_per_step = 2
motion_noise_std = 0.8
sensor_reliability = 0.8
sensor_false_pos = 0.15
compare_at_step = 5            # which step to compare filter vs smoother

rng = np.random.default_rng(7)

# simulate a true trajectory
true_positions = [(2 + motion_per_step * t) % n_cells for t in range(n_steps)]

# generate sensor observations (True = door sensed)
observations = []
lik_door = compute_likelihood(n_cells, door_positions, sensor_reliability, sensor_false_pos)
lik_no_door = compute_likelihood(n_cells, door_positions, sensor_false_pos, sensor_reliability)

for pos in true_positions:
    if pos in door_positions:
        sees_door = rng.random() < sensor_reliability
    else:
        sees_door = rng.random() < sensor_false_pos
    observations.append(sees_door)

# ── FORWARD PASS (filtering) ──
forward_beliefs = []
bel = np.ones(n_cells) / n_cells  # uniform start
for t in range(n_steps):
    # predict (skip at t=0, no prior motion)
    if t > 0:
        bel = predict(bel, motion_per_step, motion_noise_std, n_cells)
    # update
    lik = lik_door if observations[t] else lik_no_door
    bel = update(bel, lik)
    forward_beliefs.append(bel.copy())

# ── BACKWARD PASS ──
backward_messages = [None] * n_steps
beta = np.ones(n_cells) / n_cells  # uninformative start
backward_messages[-1] = beta.copy()

for t in range(n_steps - 2, -1, -1):
    # reverse predict: for each x_t, sum over x_{t+1}
    # p(x_{t+1} | u, x_t) shifted backward
    lik = lik_door if observations[t + 1] else lik_no_door
    beta_updated = beta * lik
    # "reverse convolution": shift left and blur
    beta = np.roll(beta_updated, -motion_per_step)
    if motion_noise_std > 0:
        beta = gaussian_filter1d(beta, sigma=motion_noise_std, mode='wrap')
    beta /= beta.sum()
    backward_messages[t] = beta.copy()

# ── COMBINE: smoothed = forward * backward ──
smoothed_beliefs = []
for t in range(n_steps):
    s = forward_beliefs[t] * backward_messages[t]
    s /= s.sum()
    smoothed_beliefs.append(s)

# ── PLOT comparison at the chosen step ──
cells = np.arange(n_cells)
t_show = min(compare_at_step, n_steps - 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

axes[0].bar(cells, forward_beliefs[t_show], color='steelblue', edgecolor='white', label='filter')
axes[0].axvline(true_positions[t_show], color='tomato', linestyle='--', linewidth=2, label='true position')
axes[0].set_title(f'Filtered Belief at t={t_show} (uses data 1..{t_show})', fontweight='bold')
axes[0].set_xlabel('cell')
axes[0].set_ylabel('probability')
axes[0].legend()

axes[1].bar(cells, smoothed_beliefs[t_show], color='forestgreen', edgecolor='white', label='smoothed')
axes[1].axvline(true_positions[t_show], color='tomato', linestyle='--', linewidth=2, label='true position')
axes[1].set_title(f'Smoothed Belief at t={t_show} (uses data 1..{n_steps})', fontweight='bold')
axes[1].set_xlabel('cell')
axes[1].legend()

plt.suptitle('Filtering vs Smoothing', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# entropy comparison
def entropy(p):
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

print(f"Filter  entropy at t={t_show}: {entropy(forward_beliefs[t_show]):.2f} bits")
print(f"Smoother entropy at t={t_show}: {entropy(smoothed_beliefs[t_show]):.2f} bits")
print(f"Lower entropy means more confident. The smoother is sharper because it uses future data too.")

**When to use which?**

- **Filtering** is for **real time** applications: the robot must act now, and future data
  is not yet available.
- **Smoothing** is for **offline** analysis: after the mission, you want the best possible
  estimate of the entire trajectory. Mapping and calibration often use smoothing.

## 9.5 Failure Modes

The Bayes filter is optimal **if its assumptions are correct**. When the assumptions break,
the filter can fail catastrophically. Three common failures:

1. **Wrong motion model:** the filter thinks the robot moves differently than it actually does
2. **Wrong sensor model:** the likelihood function does not match reality
3. **Violated Markov assumption:** the current state depends on more than just the previous state

Let us demonstrate the first failure mode, which is the most common in practice.

In [ ]:
# ── PARAMETERS ── change these and re-run ────
n_cells = 20
true_motion = 2           # robot actually moves 2 cells per step
assumed_motion = 3        # but the filter thinks it moves 3
motion_noise_std = 0.6
n_steps = 10
door_positions = [4, 11, 17]
sensor_reliability = 0.85
sensor_false_pos = 0.1
true_start = 2

rng = np.random.default_rng(21)

lik_door = compute_likelihood(n_cells, door_positions, sensor_reliability, sensor_false_pos)
lik_no_door = compute_likelihood(n_cells, door_positions, sensor_false_pos, sensor_reliability)

# simulate true trajectory
true_pos_list = []
pos = true_start
for t in range(n_steps):
    true_pos_list.append(pos)
    pos = (pos + true_motion) % n_cells

# run filter with WRONG motion model
bel_wrong = np.ones(n_cells) / n_cells
bel_correct = np.ones(n_cells) / n_cells
wrong_beliefs = []
correct_beliefs = []

for t in range(n_steps):
    # generate observation at true position
    tp = true_pos_list[t]
    if tp in door_positions:
        sees_door = rng.random() < sensor_reliability
    else:
        sees_door = rng.random() < sensor_false_pos
    lik = lik_door if sees_door else lik_no_door

    # predict
    if t > 0:
        bel_wrong = predict(bel_wrong, assumed_motion, motion_noise_std, n_cells)  # WRONG
        bel_correct = predict(bel_correct, true_motion, motion_noise_std, n_cells)  # CORRECT

    # update both
    bel_wrong = update(bel_wrong, lik)
    bel_correct = update(bel_correct, lik)

    wrong_beliefs.append(bel_wrong.copy())
    correct_beliefs.append(bel_correct.copy())

# plot at several time steps
plot_steps = [0, 3, 6, 9]
plot_steps = [s for s in plot_steps if s < n_steps]
fig, axes = plt.subplots(2, len(plot_steps), figsize=(4 * len(plot_steps), 7), sharey=True)
cells = np.arange(n_cells)

for i, t in enumerate(plot_steps):
    # correct model
    axes[0, i].bar(cells, correct_beliefs[t], color='forestgreen', edgecolor='white', alpha=0.8)
    axes[0, i].axvline(true_pos_list[t], color='tomato', linestyle='--', linewidth=2)
    axes[0, i].set_title(f't={t}', fontweight='bold')
    axes[0, i].set_xlabel('cell')
    if i == 0:
        axes[0, i].set_ylabel('Correct model', fontweight='bold')

    # wrong model
    axes[1, i].bar(cells, wrong_beliefs[t], color='steelblue', edgecolor='white', alpha=0.8)
    axes[1, i].axvline(true_pos_list[t], color='tomato', linestyle='--', linewidth=2)
    axes[1, i].set_xlabel('cell')
    if i == 0:
        axes[1, i].set_ylabel(f'Wrong model (assumes {assumed_motion})', fontweight='bold')

plt.suptitle(f'Failure Mode: Wrong Motion Model (true={true_motion}, assumed={assumed_motion})',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Red dashed line = true position.")
print(f"Top row (correct model): belief tracks the true position well.")
print(f"Bottom row (wrong model): belief drifts away. The filter becomes overconfident in the wrong place.")
print(f"\nThis is dangerous: the filter does not know it is wrong!")

**Takeaway:** A Bayes filter is only as good as its models. If the motion model or
sensor model is wrong, the filter can produce confident but **incorrect** beliefs.
Always validate your models against real data.

## 9.6 Capstone: Full Bayes Filter in a Circular Hallway

Now let us put it all together. A robot navigates a **circular hallway** with 30 cells.
8 of these cells have doors. The robot starts completely lost (uniform belief) and
takes 20 steps of random motion while sensing doors at each step.

Watch the belief evolve from a flat line to a sharp peak as evidence accumulates.

In [ ]:
# ── PARAMETERS ── change these and re-run ────
n_cells = 30
door_positions = [2, 6, 10, 14, 18, 22, 25, 28]
n_steps = 20
true_start = 7
sensor_reliability = 0.80
sensor_false_pos = 0.10
motion_noise_std = 0.7
seed = 42

rng = np.random.default_rng(seed)

lik_door = compute_likelihood(n_cells, door_positions, sensor_reliability, sensor_false_pos)
lik_no_door = compute_likelihood(n_cells, door_positions, sensor_false_pos, sensor_reliability)

# simulate true trajectory (random motions of 1 to 3 cells)
true_positions = [true_start]
motions = rng.integers(1, 4, size=n_steps)  # random motion 1, 2, or 3
for m in motions:
    true_positions.append((true_positions[-1] + m) % n_cells)

# generate observations
observations = []
for pos in true_positions:
    if pos in door_positions:
        observations.append(rng.random() < sensor_reliability)
    else:
        observations.append(rng.random() < sensor_false_pos)

# run the Bayes filter
belief = np.ones(n_cells) / n_cells
all_beliefs = [belief.copy()]

for t in range(n_steps):
    # predict
    motion = motions[t]
    belief = predict(belief, int(motion), motion_noise_std, n_cells)

    # update
    lik = lik_door if observations[t + 1] else lik_no_door
    belief = update(belief, lik)

    all_beliefs.append(belief.copy())

# ── ANIMATE as a grid of snapshots ──
show_steps = [0, 2, 5, 8, 12, 16, 20]
show_steps = [s for s in show_steps if s <= n_steps]
n_show = len(show_steps)

fig, axes = plt.subplots(1, n_show, figsize=(3.2 * n_show, 3.5), sharey=True)
cells = np.arange(n_cells)

for i, t in enumerate(show_steps):
    ax = axes[i]
    colors = ['orange' if c in door_positions else 'steelblue' for c in cells]
    ax.bar(cells, all_beliefs[t], color=colors, edgecolor='white', linewidth=0.3)
    ax.axvline(true_positions[t], color='tomato', linestyle='--', linewidth=2)
    ax.set_title(f't={t}', fontweight='bold')
    ax.set_xlabel('cell')
    if i == 0:
        ax.set_ylabel('probability')
    ax.set_xlim(-0.5, n_cells - 0.5)

plt.suptitle('Bayes Filter: From Lost to Located (orange bars = door cells)',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# final stats
final_est = np.argmax(all_beliefs[-1])
final_true = true_positions[-1]
print(f"After {n_steps} steps:")
print(f"  True position:      cell {final_true}")
print(f"  MAP estimate:       cell {final_est}")
print(f"  Max probability:    {all_beliefs[-1].max():.3f}")
print(f"  Entropy:            {entropy(all_beliefs[-1]):.2f} bits (started at {np.log2(n_cells):.2f} bits)")

### Belief Entropy Over Time

Entropy measures how uncertain the belief is. Maximum entropy means completely lost.
Let us track how entropy drops as the filter processes more data.

In [ ]:
entropies = [entropy(b) for b in all_beliefs]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(len(entropies)), entropies, 'o-', color='steelblue', linewidth=2, markersize=5)
ax.axhline(np.log2(n_cells), color='tomato', linestyle='--', label='max entropy (uniform)')
ax.axhline(0, color='forestgreen', linestyle='--', label='min entropy (certain)')
ax.set_xlabel('time step', fontweight='bold')
ax.set_ylabel('entropy (bits)', fontweight='bold')
ax.set_title('Belief Entropy Over Time', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print("Entropy drops quickly at first, then levels off as the robot becomes confident.")

## Exercises

**Exercise 9.1: Custom Likelihood Functions**

Instead of a binary door sensor, suppose the robot has a **range sensor** that measures
the distance to the nearest wall. The hallway has walls at cells 0, 10, and 19.
Design a likelihood function $p(z \mid x)$ where the measurement is the distance to the
nearest wall, corrupted by Gaussian noise with $\sigma = 1.5$ cells. Run the Bayes filter
with this continuous likelihood and compare convergence speed to the door sensor.

**Exercise 9.2: Multi Modal Beliefs**

Create a hallway where the door pattern is **symmetric**: doors at cells 3, 8, 13, 18
(evenly spaced by 5). Start the robot with a uniform belief and observe what happens
after many predict/update cycles. Can the filter ever resolve the ambiguity? Under
what conditions does the multi modal belief collapse to a single peak?

**Exercise 9.3: Recovery from Wrong Initial Belief**

Initialize the belief as a sharp Gaussian centered at the **wrong** cell (e.g., true
position is cell 15 but initial belief peaks at cell 5). Run the Bayes filter and
measure how many steps it takes for the filter to recover and find the correct position.
Experiment with different `sensor_reliability` values. How does sensor quality affect
recovery time?

**Exercise 9.4: Grid Resolution**

Run the Bayes filter with n_cells = 10, 20, 50, and 100 for the same physical hallway
(10 meters long). Compare: (a) localization accuracy, (b) computation time, and
(c) the sharpness of the final belief. Plot all four final beliefs on the same axes.
What is the practical trade off between resolution and cost?

**Exercise 9.5: Capstone Extension**

Extend the circular hallway capstone by adding a **kidnapped robot** event: at step 10,
teleport the robot to a random new position without telling the filter. Observe how
the filter initially gives a wrong estimate, then gradually recovers as new sensor data
overwhelms the stale belief. Add a "kidnap detector" that monitors belief entropy and
triggers a belief reset when entropy suddenly rises.

## Summary

The Bayes filter is the **recursive estimation engine** at the heart of robotics:

| Concept | Key Equation | Intuition |
|---|---|---|
| **Belief** | $\text{bel}(x_t) = p(x_t \mid z_{1:t}, u_{1:t})$ | Everything the robot knows |
| **Prediction** | $\overline{\text{bel}}(x_t) = \int p(x_t \mid u_t, x_{t-1})\,\text{bel}(x_{t-1})\,dx_{t-1}$ | Motion shifts and spreads belief |
| **Update** | $\text{bel}(x_t) = \eta\,p(z_t \mid x_t)\,\overline{\text{bel}}(x_t)$ | Measurement sharpens belief |
| **Filtering** | Uses data up to time $t$ | Real time estimation |
| **Smoothing** | Uses all data (past and future) | Offline, sharper estimates |

In the next chapter, we will see how the same estimation problem can be viewed through
the lens of **optimization**, connecting Bayesian estimation to least squares.